In [13]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-opus-4-1-20250805"

In [14]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    print(message)              # Entire response
    print(message.content)      # Content blocks
    print(message.stop_reason)


    # return message

    if not message.content:
        raise RuntimeError("Model returned no content")
    return message.content[0].text



In [50]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluation the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [51]:
dataset = generate_dataset()
dataset

Message(id='msg_011Cd2RvMAevUGrD69LkKsE9', container=None, content=[TextBlock(citations=None, text='\n[\n    {\n        "task": "Write a Python function that takes an S3 bucket name and object key as parameters and generates a pre-signed URL valid for 1 hour using boto3",\n        "format": "python",\n        "solution_criteria": "Function correctly uses boto3 client, includes proper parameters (Bucket, Key), sets expiration to 3600 seconds, handles the generate_presigned_url method, and returns a valid URL string"\n    },\n    {\n        "task": "Create a JSON policy document that allows read-only access to all objects in an S3 bucket named \'my-data-bucket\' for any AWS principal",\n        "format": "json",\n        "solution_criteria": "Valid IAM policy structure with Version 2012-10-17, contains Allow effect, includes s3:GetObject action, correct resource ARN format (arn:aws:s3:::my-data-bucket/*), and Principal set to \'*\' or AWS \'*\'"\n    },\n    {\n        "task": "Write a r

[{'task': 'Write a Python function that takes an S3 bucket name and object key as parameters and generates a pre-signed URL valid for 1 hour using boto3',
  'format': 'python',
  'solution_criteria': 'Function correctly uses boto3 client, includes proper parameters (Bucket, Key), sets expiration to 3600 seconds, handles the generate_presigned_url method, and returns a valid URL string'},
 {'task': "Create a JSON policy document that allows read-only access to all objects in an S3 bucket named 'my-data-bucket' for any AWS principal",
  'format': 'json',
  'solution_criteria': "Valid IAM policy structure with Version 2012-10-17, contains Allow effect, includes s3:GetObject action, correct resource ARN format (arn:aws:s3:::my-data-bucket/*), and Principal set to '*' or AWS '*'"},
 {'task': 'Write a regular expression that matches and validates AWS EC2 instance IDs, which follow the pattern: i- followed by either 8 or 17 hexadecimal characters',
  'format': 'regex',
  'solution_criteria': 

In [52]:
with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

In [39]:
def run_prompt(test_case):
    """Mergers the prompt and test case input, then return result"""
    prompt = f"""Please solve the following task:
    {test_case["task"]}


    * Respond only with Python, Json, or plan Regex
    * Do not add any comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output= chat(messages, stop_sequences=["```"])
    return output

In [53]:
def grade_by_model(test_case, output):
    # Function to grade a test case + output using a model

    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Criteria you should use to evaluate the solution
    <criteria>
    {test_case['solution_criteria']}
    </criteris>

    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
        """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [54]:
import re 
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    try:
        re.compile(text.strip()) 
        return 10
    except re.error: 
        return 0  
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)    

In [55]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade['score']
    reasoning = model_grade['reasoning']

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [56]:
from statistics import mean
def run_eval(dataset):
    """Load the dataset and calls run-test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score =  mean([result['score'] for result in results])
    print("Average Score: ",average_score)
    return results    

In [57]:
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)    

Message(id='msg_011Cd2S4Ykdqe2DUjVhWpnoh', container=None, content=[TextBlock(citations=None, text="\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef generate_presigned_url(bucket_name, object_key):\n    s3_client = boto3.client('s3')\n    \n    try:\n        response = s3_client.generate_presigned_url(\n            'get_object',\n            Params={'Bucket': bucket_name, 'Key': object_key},\n            ExpiresIn=3600\n        )\n    except ClientError as e:\n        return None\n    \n    return response\n", type='text')], model='claude-opus-4-1-20250805', role='assistant', stop_details=None, stop_reason='stop_sequence', stop_sequence='```', type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=79, output_tokens=127, server_tool_use=None, service_tier='standard'))
[TextBlock(citations=None, text="\nimport

In [ ]:
print(results)

[{'output': 'Here\'s a Python function that generates a pre-signed URL for an AWS S3 object:\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError, NoCredentialsError\nfrom datetime import datetime, timedelta\n\ndef generate_presigned_url(bucket_name, object_key, expiration=3600):\n    """\n    Generate a pre-signed URL for an S3 object that expires in 1 hour by default.\n    \n    Parameters:\n    -----------\n    bucket_name : str\n        The name of the S3 bucket\n    object_key : str\n        The key (path) of the object in the S3 bucket\n    expiration : int, optional\n        Time in seconds for the pre-signed URL to remain valid (default: 3600 = 1 hour)\n    \n    Returns:\n    --------\n    str or None\n        The pre-signed URL as a string if successful, None if an error occurs\n    \n    Examples:\n    ---------\n    >>> url = generate_presigned_url(\'my-bucket\', \'path/to/file.pdf\')\n    >>> print(url)\n    https://my-bucket.s3.amazonaws.com/path/to/fil

In [58]:
print(json.dumps(results,indent=2))

[
  {
    "output": "\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef generate_presigned_url(bucket_name, object_key):\n    s3_client = boto3.client('s3')\n    \n    try:\n        response = s3_client.generate_presigned_url(\n            'get_object',\n            Params={'Bucket': bucket_name, 'Key': object_key},\n            ExpiresIn=3600\n        )\n    except ClientError as e:\n        return None\n    \n    return response\n",
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket name and object key as parameters and generates a pre-signed URL valid for 1 hour using boto3",
      "format": "python",
      "solution_criteria": "Function correctly uses boto3 client, includes proper parameters (Bucket, Key), sets expiration to 3600 seconds, handles the generate_presigned_url method, and returns a valid URL string"
    },
    "score": 8.5,
    "reasoning": "The function correctly implements the core requirements - it uses boto3 properly, 